# 07 — Executive Sustainability Model
**Levante Ferries · portfolio ficticio · 2026**

Ejecutar desde la raíz del proyecto o desde `notebooks`. Cada tabla se calcula sobre CSV regenerables.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
import numpy as np
from model import assess,aggregate,calculate,optimize,FUELEU_LIMIT
DATA=ROOT/'data'; OUT=ROOT/'outputs'
voyages=pd.read_csv(DATA/'voyages_2026_synthetic.csv')
print('Raíz:',ROOT.name,'| viajes:',len(voyages),'| periodo:',voyages.date.min(),voyages.date.max())

Raíz: levante_ferries_sustainability | viajes: 4860 | periodo: 2026-01-01 2026-08-31


## Propósito
Unir margen, emisiones, capacidad y energía en un marco de decisión. Integración conceptual: P1 aporta fiabilidad/ocupación; P2 una demanda prevista; P3 precios sujetos a elasticidad; P5 costes/margen y caja. Este proyecto usa valores sintéticos propios y no importa outputs de esos repositorios.

In [2]:
f=assess(voyages); r=aggregate(f,['route']); sol=__import__('json').loads((OUT/'optimization_summary.json').read_text())
r['zone']=np.where((r.margin_eur>=r.margin_eur.median())&(r.kg_co2e_per_pax<=r.kg_co2e_per_pax.median()),'Alto margen / baja intensidad',np.where((r.margin_eur<r.margin_eur.median())&(r.kg_co2e_per_pax>r.kg_co2e_per_pax.median()),'Prioridad de intervención','Revisar contexto'))
print(r[['route','margin_eur','ttw_co2e_kg','kg_co2e_per_pax','zone']].round(1).to_string(index=False))

             route  margin_eur  ttw_co2e_kg  kg_co2e_per_pax                          zone
Algeciras – Tánger  11738138.2    1149482.1              4.0              Revisar contexto
 Barcelona – Ibiza  32970181.3    6290938.4             17.9              Revisar contexto
 Barcelona – Palma  28381469.8    4497751.6             12.5              Revisar contexto
Dénia – Formentera  11425998.9    1938490.1              8.6              Revisar contexto
     Dénia – Ibiza  32429568.0    4218581.9              7.3 Alto margen / baja intensidad
  Dénia – Mallorca  23195585.5    4731537.7             16.2     Prioridad de intervención
   Valencia – Orán  30896124.0    6678600.5             21.6              Revisar contexto
  Valencia – Palma  64508570.3   10850348.6             14.7              Revisar contexto


In [3]:
print('Margen base €:',round(f.margin_eur.sum()),'| reducción optimizada %:',round(100*(1-sol['optimized_ttw_t']/sol['baseline_ttw_t']),2),'| Δcoste €:',round(sol['optimized_cost_eur']-sol['baseline_cost_eur']))
print('Decisión: validar horarios y combustible certificado antes de contratar o implantar; contrastar con forecast y pricing sin asumir enlace de registros.')

Margen base €: 235545636 | reducción optimizada %: 13.39 | Δcoste €: -1231988
Decisión: validar horarios y combustible certificado antes de contratar o implantar; contrastar con forecast y pricing sin asumir enlace de registros.


**Alcance ejecutivo:** las recomendaciones son pruebas de escenarios sintéticos, no autorizaciones de inversión ni declaraciones certificadas de cumplimiento.